# QAI PR 3 – 5-Qubit Quantum Fourier Transform (QFT)

# QAI Practical

This notebook recreates the practical shown in the supplied PDF. Run cells from top to bottom in Google Colab or a local Jupyter environment. The install cell is intended for Colab/Jupyter.

### 1. Install packages

Installs Qiskit and Qiskit Aer for circuit construction and simulation, plus Matplotlib for the result histogram.

In [ ]:
!pip -q install qiskit qiskit-aer matplotlib


### 2. Import libraries

Imports quantum and classical registers, the circuit and transpiler, the QFT circuit component, AerSimulator, and plotting tools.

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

print("Libraries imported successfully.")


### 3. Create five-qubit registers

Creates five qubits and five classical bits, then combines them in a circuit. The classical register stores the five measured output bits.

In [ ]:
q = QuantumRegister(5, 'q')
c = ClassicalRegister(5, 'c')
circuit = QuantumCircuit(q, c)

print("5-Qubit Quantum Circuit Created Successfully.")


### 4. Encode input state 10101

The loop reads the string in reverse because Qiskit numbers qubits from least significant to most significant. An X gate is applied wherever the corresponding input bit is 1, preparing the computational-basis input state.

In [ ]:
# Encode the input state 10101 as shown in the practical.
# Qiskit uses little-endian qubit indexing: the rightmost bit is q[0].
for qubit, bit in enumerate(reversed("10101")):
    if bit == "1":
        circuit.x(q[qubit])

circuit.barrier()
print("Input state 10101 encoded successfully.")


### 5. Apply the Quantum Fourier Transform

Constructs a five-qubit forward QFT (inverse=False). do_swaps=True includes the final bit-reversal swaps. compose inserts the QFT operations into the existing circuit.

In [ ]:
qft = QFT(
    num_qubits=5,
    approximation_degree=0,
    do_swaps=True,
    inverse=False,
    insert_barriers=True,
    name='QFT'
)

circuit.compose(qft, qubits=q, inplace=True)

print("5-Qubit Quantum Fourier Transform applied successfully.")


### 6. Measure and display circuit

Measures each qubit into its matching classical bit, then draws the circuit so the QFT and measurement operations can be inspected.

In [ ]:
circuit.measure(q, c)

print("All 5 qubits measured successfully.")
print(circuit.draw())


### 7. Decompose and simulate

Decomposes the QFT component into lower-level gates, transpiles the circuit for AerSimulator, and runs 1000 shots. get_counts() collects the measured output strings.

In [ ]:
# Decompose QFT into basic gates, then simulate.
simulation_circuit = circuit.decompose(reps=5)
simulator = AerSimulator()
compiled_circuit = transpile(simulation_circuit, simulator)

job = simulator.run(compiled_circuit, shots=1000)
result = job.result()
counts = result.get_counts()

print("QFT Simulation Completed.")
print("QFT Output:")
print(counts)


### 8. Visualize QFT output

Plots the frequency of each measured five-bit output. The distribution is an empirical result from the simulator's 1000 runs.

In [ ]:
fig = plot_histogram(counts, figsize=(8, 4))
plt.show()
